# Prototype Computing the Jacobian with AD

This notebook implements a prototype for computing the derivatives of a square problem via automatic differentiation. See https://github.com/Pyomo/pyomo/issues/3564

In [2]:
import pyomo.environ as pyo
import idaes # load solvers

In [3]:
# Define a simple linear optimization problem
# This problem has no degrees of freedom and only linear equality constraints.
model = pyo.ConcreteModel()

# Define variables
model.x = pyo.Var(domain=pyo.NonNegativeReals)
model.y = pyo.Var(domain=pyo.NonNegativeReals)

# Define objective function
model.obj = pyo.Objective(expr=3 * model.x + 4 * model.y, sense=pyo.maximize)

# Define constraints
model.con1 = pyo.Constraint(expr=2 * model.x + model.y == 8)
model.con2 = pyo.Constraint(expr=model.x + 2 * model.y == 6)

# Solve the problem
solver = pyo.SolverFactory('ipopt')
solver.solve(model)

# Print results
print(f"x = {model.x.value}")
print(f"y = {model.y.value}")
print(f"Objective value = {model.obj()}")

x = 3.333333333333333
y = 1.3333333333333333
Objective value = 15.333333333333332


In [6]:
from pyomo.core.expr.calculus.diff_with_pyomo import reverse_sd

for c in model.component_objects(pyo.Constraint, active=True):
    print(f"Considering constraint: {c.name}")
    for index in c:
        print(f"  {index}: {c[index].expr}")

        for k, v in reverse_sd(c).items():
            print(f'derivative of {str(c)} with respect to {str(k)} is {str(v)}')
    

Considering constraint: con1
  None: 2*x + y  ==  8
derivative of con1 with respect to con1 is 1
Considering constraint: con2
  None: x + 2*y  ==  6
derivative of con2 with respect to con2 is 1


In [15]:
for k, v in reverse_sd(model.con1).items():
    print(str(k), str(v))

con1 1
